# AI-BASED NIDS - Layer 2: Anomaly Detection

This notebook trains two models on **normal network traffic**:
1. **Autoencoder (Deep Learning)**
2. **Isolation Forest (Machine Learning)**

Both are used to learn the baseline behavior and detect unknown attacks.

In [ ]:
# 1. Setup & Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
import joblib

In [ ]:
# 2. Load Data
# Upload your CSV file to Colab or mount Google Drive
# dataset_path = '/content/drive/MyDrive/AI-NIDS/normal_traffic.csv'
dataset_path = 'normal_traffic.csv' # Example path

try:
    df = pd.read_csv(dataset_path)
    print(f"Loaded dataset with {len(df)} records.")
except FileNotFoundError:
    print("Error: Dataset not found. Please upload 'normal_traffic.csv'.")

In [ ]:
# 3. Preprocessing
# We only need numerical features for the Autoencoder
# Drop non-numeric columns like IPs, timestamps if present
# Example: only keep useful flow features
# feature_cols = ['duration', 'src_bytes', 'dst_bytes', 'protocol_num', ...]
# df = df[feature_cols]

# Normalize data (Critical for Autoencoders)
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df.select_dtypes(include=[np.number]))

# Convert to PyTorch Tensor
tensor_x = torch.Tensor(data_scaled)

# Create Dataset & Loader
train_data, val_data = train_test_split(tensor_x, test_size=0.2, random_state=42)
train_loader = DataLoader(TensorDataset(train_data, train_data), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(val_data, val_data), batch_size=32, shuffle=False)

input_dim = data_scaled.shape[1]
print(f"Input Features: {input_dim}")

In [ ]:
# 4. Define Autoencoder Model
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
            nn.Sigmoid()  # Use Sigmoid if data is 0-1 (MinMaxScaled)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

model = Autoencoder(input_dim=input_dim)
criterion = nn.MSELoss() # Reconstruction Error
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

In [ ]:
# 5. Train Autoencoder
num_epochs = 20
history = {'train_loss': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for data in train_loader:
        inputs, _ = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for data in val_loader:
            inputs, _ = data
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            val_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

In [ ]:
# 6. Plot Training Loss
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.title("Autoencoder Training Loss")
plt.show()

In [ ]:
# 7. Save Autoencoder Model
torch.save(model.state_dict(), 'autoencoder.pth')
print("Model saved as autoencoder.pth")

In [ ]:
# 8. Train Isolation Forest (Model 2)
# Isolation Forest needs 2D array (numpy), which we already have in 'data_scaled'
print("Training Isolation Forest...")
clf = IsolationForest(contamination=0.01, random_state=42) # contamination = expected outlier/noise rate
clf.fit(data_scaled)

# Save Model
joblib.dump(clf, 'isolation_forest.joblib')
print("Isolation Forest saved as isolation_forest.joblib")